# Portable execution configuration

This public copy was prepared from the executed JPCO revision notebook.

Machine-specific absolute paths and saved execution outputs were removed.
The original numerical summaries are provided separately in this revision
package.

By default, the notebook treats its current working directory as the
revision root. If required, set the following environment variables before
starting Jupyter:

- `GW_GLITCH_REVISION_ROOT`: directory containing the revision scripts and
  result subdirectories;
- `GW_GLITCH_PROJECT_ROOT`: parent `GW_Glitch` project directory containing
  historical processed datasets;
- `GW_GLITCH_RESEARCH_ROOT`: parent research directory;
- `GW_GLITCH_IMAGE_PROJECT_ROOT`: archived Gravity Spy image-project root.

The full source datasets, trained checkpoints, prediction arrays, and local
execution logs are not stored in Git.


In [ ]:
import os
from pathlib import Path

PORTABLE_REVISION_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_REVISION_ROOT",
        Path.cwd(),
    )
).expanduser().resolve()

PORTABLE_PROJECT_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_PROJECT_ROOT",
        PORTABLE_REVISION_ROOT.parent.parent,
    )
).expanduser().resolve()

PORTABLE_RESEARCH_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_RESEARCH_ROOT",
        PORTABLE_PROJECT_ROOT.parent,
    )
).expanduser().resolve()

PORTABLE_DESKTOP_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_DESKTOP_ROOT",
        PORTABLE_RESEARCH_ROOT.parent,
    )
).expanduser().resolve()

PORTABLE_IMAGE_PROJECT_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_IMAGE_PROJECT_ROOT",
        PORTABLE_RESEARCH_ROOT / "Glitch_Image",
    )
).expanduser().resolve()

print("Portable revision root:", PORTABLE_REVISION_ROOT)
print("Portable project root:", PORTABLE_PROJECT_ROOT)
print("Portable research root:", PORTABLE_RESEARCH_ROOT)
print("Portable image-project root:", PORTABLE_IMAGE_PROJECT_ROOT)


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import sys

import pandas as pd

REVISION_ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
).resolve()

RESEARCH_ROOT = Path(
    str(PORTABLE_RESEARCH_ROOT)
).resolve()

DATASET_DIR = (
    REVISION_ROOT
    / "jpco_o3_splits_v1"
)

OUTPUT_DIR = (
    REVISION_ROOT
    / "image_comparison_audit_v1"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_PATH = (
    DATASET_DIR / "manifest.json"
)

PRIMARY_PATH = (
    DATASET_DIR / "o3_primary.csv.gz"
)

SPLIT_SUMMARY_PATH = (
    DATASET_DIR / "split_summary.csv"
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print(
    "Notebook working directory:",
    Path.cwd(),
)
print("Research root:", RESEARCH_ROOT)
print("Revision root:", REVISION_ROOT)
print("Audit output:", OUTPUT_DIR)

required_paths = [
    REVISION_ROOT,
    RESEARCH_ROOT,
    MANIFEST_PATH,
    PRIMARY_PATH,
    SPLIT_SUMMARY_PATH,
]

for path in required_paths:
    assert path.exists(), (
        f"Required path is missing: {path}"
    )
    print("FOUND:", path)

print()
print(
    "IMAGE-COMPARISON AUDIT "
    "ENVIRONMENT VERIFIED"
)

In [ ]:
EXPECTED_HASHES = {
    "manifest.json": (
        "f1eedfaa282d6ae90e1567d271f9baff"
        "06c62ba233aff5890ae5dad85cfe9605"
    ),
    "o3_primary.csv.gz": (
        "9d7d7d217563ccb121eba466fc442d83"
        "d7f25495e6e5ba40ea9a0962114829c6"
    ),
    "split_summary.csv": (
        "61de488a44179ca86205ffcb45e98baa"
        "9eb2d9a7b52a1ab13cf2845368f88d01"
    ),
}

reference_paths = {
    "manifest.json": MANIFEST_PATH,
    "o3_primary.csv.gz": PRIMARY_PATH,
    "split_summary.csv": (
        SPLIT_SUMMARY_PATH
    ),
}

observed_hashes = {}

for name, path in (
    reference_paths.items()
):
    observed = sha256_file(path)
    expected = EXPECTED_HASHES[name]

    print(name)
    print("  observed:", observed)
    print("  expected:", expected)

    assert observed == expected, (
        f"Frozen input changed: {name}"
    )

    observed_hashes[name] = observed

manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

print()
print("Frozen tabular rows:", manifest["primary_rows"])
print("Target:", manifest["target"])
print(
    "Expected classes:",
    24,
)
print()
print(
    "FROZEN TABULAR REFERENCE VERIFIED"
)

In [ ]:
import os
from pathlib import Path

DIRECTORY_KEYWORDS = [
    "glitch",
    "gravityspy",
    "gravity_spy",
    "image",
    "resnet",
    "vision",
    "vit",
    "robust",
]

PRUNE_DIRECTORIES = {
    ".git",
    ".ipynb_checkpoints",
    "__pycache__",
    "node_modules",
    "site-packages",
    "env",
    "venv",
    ".venv",
    "anaconda3",
    "miniconda3",
    "Library",
}

MAX_DEPTH = 5

candidate_directories = set()

research_root_depth = len(
    RESEARCH_ROOT.parts
)

for current, directories, files in (
    os.walk(RESEARCH_ROOT)
):
    current_path = Path(current)
    depth = (
        len(current_path.parts)
        - research_root_depth
    )

    directories[:] = [
        name
        for name in directories
        if name not in PRUNE_DIRECTORIES
        and not name.startswith(".")
    ]

    if depth >= MAX_DEPTH:
        directories[:] = []

    lowered = (
        current_path.name
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    if any(
        keyword in lowered
        for keyword in DIRECTORY_KEYWORDS
    ):
        candidate_directories.add(
            current_path.resolve()
        )

candidate_directories = sorted(
    candidate_directories,
    key=lambda path: (
        len(path.parts),
        str(path).lower(),
    ),
)

print(
    "Candidate directories found:",
    len(candidate_directories),
)

for index, path in enumerate(
    candidate_directories,
    start=1,
):
    print(f"{index:3d}. {path}")

discovery_path = (
    OUTPUT_DIR
    / "candidate_directories.txt"
)

discovery_path.write_text(
    "\n".join(
        str(path)
        for path in candidate_directories
    )
    + "\n",
    encoding="utf-8",
)

print()
print("SAVED:", discovery_path)
print()
print(
    "BOUNDED PROJECT DISCOVERY COMPLETE"
)

In [ ]:
PROJECT_MARKERS = {
    "requirements.txt",
    "environment.yml",
    "environment.yaml",
    "pyproject.toml",
    "setup.py",
    "README.md",
    "README.rst",
}

MODEL_EXTENSIONS = {
    ".pt",
    ".pth",
    ".ckpt",
    ".h5",
    ".keras",
    ".onnx",
    ".pkl",
    ".joblib",
}

IMAGE_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".webp",
    ".tif",
    ".tiff",
}

TABLE_EXTENSIONS = {
    ".csv",
    ".tsv",
    ".parquet",
    ".json",
    ".jsonl",
}

likely_roots = []

for directory in candidate_directories:
    try:
        immediate_entries = list(
            directory.iterdir()
        )
    except (
        PermissionError,
        FileNotFoundError,
        OSError,
    ):
        continue

    immediate_files = {
        entry.name
        for entry in immediate_entries
        if entry.is_file()
    }

    python_files = [
        entry
        for entry in immediate_entries
        if entry.is_file()
        and entry.suffix.lower() == ".py"
    ]

    notebooks = [
        entry
        for entry in immediate_entries
        if entry.is_file()
        and entry.suffix.lower() == ".ipynb"
    ]

    model_files = [
        entry
        for entry in immediate_entries
        if entry.is_file()
        and entry.suffix.lower()
        in MODEL_EXTENSIONS
    ]

    marker_files = sorted(
        immediate_files
        & PROJECT_MARKERS
    )

    score = 0
    score += 3 * bool(marker_files)
    score += 2 * bool(python_files)
    score += 2 * bool(notebooks)
    score += 2 * bool(model_files)

    if score >= 2:
        likely_roots.append(
            {
                "path": str(directory),
                "score": score,
                "markers": marker_files,
                "immediate_python_files": (
                    len(python_files)
                ),
                "immediate_notebooks": (
                    len(notebooks)
                ),
                "immediate_model_files": (
                    len(model_files)
                ),
            }
        )

likely_roots = sorted(
    likely_roots,
    key=lambda row: (
        -row["score"],
        row["path"].lower(),
    ),
)

print(
    "Likely project roots:",
    len(likely_roots),
)

for row in likely_roots:
    print()
    print(row["path"])
    print("  score:", row["score"])
    print("  markers:", row["markers"])
    print(
        "  immediate Python files:",
        row["immediate_python_files"],
    )
    print(
        "  immediate notebooks:",
        row["immediate_notebooks"],
    )
    print(
        "  immediate model files:",
        row["immediate_model_files"],
    )

likely_roots_path = (
    OUTPUT_DIR
    / "likely_project_roots.json"
)

likely_roots_path.write_text(
    json.dumps(
        likely_roots,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print()
print("SAVED:", likely_roots_path)
print()
print(
    "LIKELY PROJECT ROOT "
    "DISCOVERY COMPLETE"
)

In [ ]:
from collections import Counter

INVENTORY_MAX_DEPTH = 4

inventory_records = []

for project in likely_roots:
    project_root = Path(
        project["path"]
    )

    root_depth = len(
        project_root.parts
    )

    counts = Counter()
    examples = {
        "python": [],
        "notebook": [],
        "checkpoint": [],
        "table": [],
        "image": [],
    }

    for current, directories, files in (
        os.walk(project_root)
    ):
        current_path = Path(current)
        depth = (
            len(current_path.parts)
            - root_depth
        )

        directories[:] = [
            name
            for name in directories
            if name not in PRUNE_DIRECTORIES
            and not name.startswith(".")
        ]

        if depth >= INVENTORY_MAX_DEPTH:
            directories[:] = []

        for filename in files:
            path = current_path / filename
            suffix = path.suffix.lower()

            if suffix == ".py":
                category = "python"
            elif suffix == ".ipynb":
                category = "notebook"
            elif suffix in MODEL_EXTENSIONS:
                category = "checkpoint"
            elif suffix in TABLE_EXTENSIONS:
                category = "table"
            elif suffix in IMAGE_EXTENSIONS:
                category = "image"
            else:
                continue

            counts[category] += 1

            if len(examples[category]) < 20:
                examples[category].append(
                    str(
                        path.relative_to(
                            project_root
                        )
                    )
                )

    record = {
        "project_root": (
            str(project_root)
        ),
        "counts": dict(counts),
        "examples": examples,
    }

    inventory_records.append(record)

print(
    "Project inventories:",
    len(inventory_records),
)

for record in inventory_records:
    print()
    print("=" * 80)
    print(record["project_root"])
    print("Counts:", record["counts"])

    for category in [
        "python",
        "notebook",
        "checkpoint",
        "table",
        "image",
    ]:
        sample = record[
            "examples"
        ][category]

        if sample:
            print(
                f"  {category} examples:"
            )

            for filename in sample[:10]:
                print("   ", filename)

inventory_path = (
    OUTPUT_DIR
    / "candidate_artifact_inventory.json"
)

inventory_path.write_text(
    json.dumps(
        inventory_records,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print()
print("SAVED:", inventory_path)
print()
print(
    "SHALLOW ARTIFACT INVENTORY COMPLETE"
)
print(
    "NO IMAGE MODEL TRAINING OR "
    "TEST EVALUATION WAS PERFORMED"
)

In [ ]:
from pathlib import Path
import json
import os

IMAGE_PROJECT = Path(
    str(PORTABLE_IMAGE_PROJECT_ROOT)
).resolve()

IMAGE_DATASET = (
    IMAGE_PROJECT / "dataset"
)

assert IMAGE_PROJECT.is_dir()
assert IMAGE_DATASET.is_dir()

candidate_record = {
    "selected_candidate": (
        str(IMAGE_PROJECT)
    ),
    "selection_reason": (
        "Archived Gravity Spy image-model "
        "project containing image data, "
        "experiment runs, and ViT outputs."
    ),
    "excluded_as_primary_baseline": {
        str(
            Path(
                str(PORTABLE_RESEARCH_ROOT / "GlitchGen")
            )
        ): (
            "Synthetic image-generation "
            "package rather than the archived "
            "discriminative classifier."
        ),
        str(
            Path(
                str(PORTABLE_PROJECT_ROOT)
            )
        ): (
            "Primary tabular project; image "
            "files are predominantly result "
            "figures rather than classifier "
            "inputs."
        ),
    },
    "training_performed": False,
    "test_evaluated": False,
}

candidate_record_path = (
    OUTPUT_DIR
    / "selected_image_project.json"
)

candidate_record_path.write_text(
    json.dumps(
        candidate_record,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print("Selected project:", IMAGE_PROJECT)
print("Dataset directory:", IMAGE_DATASET)
print("SAVED:", candidate_record_path)
print()
print(
    "IMAGE PROJECT CANDIDATE FROZEN"
)

In [ ]:
RELEVANT_SUFFIXES = {
    ".py",
    ".ipynb",
    ".csv",
    ".tsv",
    ".json",
    ".jsonl",
    ".yaml",
    ".yml",
    ".txt",
    ".log",
    ".md",
    ".pt",
    ".pth",
    ".ckpt",
    ".h5",
    ".keras",
    ".onnx",
    ".pkl",
    ".joblib",
    ".npz",
    ".npy",
}

PRUNE_NAMES = {
    ".git",
    ".ipynb_checkpoints",
    "__pycache__",
    "node_modules",
    "site-packages",
    ".venv",
    "venv",
    "env",
}

inventory = []

for current, directories, files in (
    os.walk(IMAGE_PROJECT)
):
    directories[:] = [
        name
        for name in directories
        if name not in PRUNE_NAMES
        and not name.startswith(".")
    ]

    current_path = Path(current)

    for filename in files:
        path = current_path / filename

        if (
            path.suffix.lower()
            not in RELEVANT_SUFFIXES
        ):
            continue

        inventory.append(
            {
                "relative_path": str(
                    path.relative_to(
                        IMAGE_PROJECT
                    )
                ),
                "suffix": (
                    path.suffix.lower()
                ),
                "size_bytes": (
                    path.stat().st_size
                ),
            }
        )

inventory = sorted(
    inventory,
    key=lambda row: (
        row["suffix"],
        row["relative_path"].lower(),
    ),
)

print(
    "Relevant non-image artifacts:",
    len(inventory),
)

suffix_counts = {}

for record in inventory:
    suffix = record["suffix"]

    suffix_counts[suffix] = (
        suffix_counts.get(suffix, 0)
        + 1
    )

print("Counts by suffix:")

for suffix, count in sorted(
    suffix_counts.items()
):
    print(f"  {suffix}: {count}")

print()
print("First 150 relevant artifacts:")

for record in inventory[:150]:
    print(
        f"{record['relative_path']} "
        f"({record['size_bytes']:,} bytes)"
    )

inventory_path = (
    OUTPUT_DIR
    / "image_project_nonimage_inventory.json"
)

inventory_path.write_text(
    json.dumps(
        inventory,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print()
print("SAVED:", inventory_path)
print()
print(
    "NON-IMAGE PROJECT INVENTORY COMPLETE"
)

In [ ]:
IMAGE_SUFFIXES = {
    ".png",
    ".jpg",
    ".jpeg",
    ".webp",
    ".tif",
    ".tiff",
}

POSSIBLE_IMAGE_ROOTS = [
    IMAGE_DATASET / "data_mini" / "images",
    IMAGE_DATASET / "data_mini" / "synth" / "images",
    IMAGE_DATASET / "images",
    IMAGE_DATASET / "data" / "images",
]

image_root_records = []

for image_root in (
    POSSIBLE_IMAGE_ROOTS
):
    if not image_root.is_dir():
        continue

    print()
    print("=" * 80)
    print("IMAGE ROOT:", image_root)

    immediate_directories = sorted(
        [
            path
            for path in image_root.iterdir()
            if path.is_dir()
        ],
        key=lambda path: path.name.lower(),
    )

    class_counts = {}

    for directory in (
        immediate_directories
    ):
        count = sum(
            1
            for path in directory.rglob("*")
            if path.is_file()
            and path.suffix.lower()
            in IMAGE_SUFFIXES
        )

        class_counts[
            directory.name
        ] = count

    loose_images = sum(
        1
        for path in image_root.iterdir()
        if path.is_file()
        and path.suffix.lower()
        in IMAGE_SUFFIXES
    )

    total_images = (
        sum(class_counts.values())
        + loose_images
    )

    print(
        "Immediate class directories:",
        len(class_counts),
    )
    print("Loose images:", loose_images)
    print("Total images:", total_images)

    for class_name, count in (
        class_counts.items()
    ):
        print(
            f"  {class_name}: {count:,}"
        )

    image_root_records.append(
        {
            "image_root": str(
                image_root
            ),
            "class_directory_count": (
                len(class_counts)
            ),
            "class_counts": (
                class_counts
            ),
            "loose_images": (
                loose_images
            ),
            "total_images": (
                total_images
            ),
        }
    )

image_root_report_path = (
    OUTPUT_DIR
    / "image_root_class_inventory.json"
)

image_root_report_path.write_text(
    json.dumps(
        image_root_records,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print()
print(
    "SAVED:",
    image_root_report_path,
)
print()
print(
    "IMAGE CLASS-DIRECTORY "
    "AUDIT COMPLETE"
)

In [ ]:
TABLE_SUFFIXES = {
    ".csv",
    ".tsv",
    ".parquet",
    ".json",
    ".jsonl",
}

METADATA_KEYWORDS = {
    "meta",
    "label",
    "class",
    "split",
    "manifest",
    "catalog",
    "url",
    "index",
    "event",
    "gravity",
    "id",
    "test",
    "train",
    "valid",
}

metadata_candidates = []

for path in IMAGE_PROJECT.rglob("*"):
    if not path.is_file():
        continue

    if (
        path.suffix.lower()
        not in TABLE_SUFFIXES
    ):
        continue

    lowered = path.name.lower()

    matched_keywords = sorted(
        keyword
        for keyword in METADATA_KEYWORDS
        if keyword in lowered
    )

    if not matched_keywords:
        continue

    metadata_candidates.append(
        {
            "path": str(path),
            "relative_path": str(
                path.relative_to(
                    IMAGE_PROJECT
                )
            ),
            "size_bytes": (
                path.stat().st_size
            ),
            "matched_keywords": (
                matched_keywords
            ),
        }
    )

metadata_candidates = sorted(
    metadata_candidates,
    key=lambda row: (
        -len(row["matched_keywords"]),
        row["relative_path"].lower(),
    ),
)

print(
    "Candidate metadata/split files:",
    len(metadata_candidates),
)

for index, record in enumerate(
    metadata_candidates,
    start=1,
):
    print()
    print(
        f"{index:3d}. "
        f"{record['relative_path']}"
    )
    print(
        "     size:",
        f"{record['size_bytes']:,}",
    )
    print(
        "     keywords:",
        record["matched_keywords"],
    )

metadata_candidate_path = (
    OUTPUT_DIR
    / "metadata_table_candidates.json"
)

metadata_candidate_path.write_text(
    json.dumps(
        metadata_candidates,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print()
print(
    "SAVED:",
    metadata_candidate_path,
)
print()
print(
    "METADATA CANDIDATE "
    "DISCOVERY COMPLETE"
)

In [ ]:
CSV_SIZE_LIMIT = (
    500 * 1024 * 1024
)

schema_records = []

for record in metadata_candidates:
    path = Path(record["path"])

    if path.suffix.lower() not in {
        ".csv",
        ".tsv",
    }:
        continue

    if (
        path.stat().st_size
        > CSV_SIZE_LIMIT
    ):
        schema_records.append(
            {
                "relative_path": (
                    record["relative_path"]
                ),
                "status": (
                    "skipped_over_size_limit"
                ),
                "size_bytes": (
                    path.stat().st_size
                ),
            }
        )
        continue

    separator = (
        "\t"
        if path.suffix.lower() == ".tsv"
        else ","
    )

    try:
        sample = pd.read_csv(
            path,
            sep=separator,
            nrows=5,
        )

        columns = (
            sample.columns.tolist()
        )

        label_like_columns = [
            column
            for column in columns
            if any(
                keyword in column.lower()
                for keyword in [
                    "label",
                    "class",
                    "target",
                    "category",
                    "type",
                ]
            )
        ]

        id_like_columns = [
            column
            for column in columns
            if any(
                keyword in column.lower()
                for keyword in [
                    "id",
                    "event",
                    "gravityspy",
                    "sample",
                    "filename",
                    "file",
                    "url",
                    "gps",
                    "time",
                ]
            )
        ]

        split_like_columns = [
            column
            for column in columns
            if any(
                keyword in column.lower()
                for keyword in [
                    "split",
                    "train",
                    "test",
                    "valid",
                    "fold",
                    "partition",
                ]
            )
        ]

        schema_records.append(
            {
                "relative_path": (
                    record["relative_path"]
                ),
                "status": "readable",
                "size_bytes": (
                    path.stat().st_size
                ),
                "columns": columns,
                "label_like_columns": (
                    label_like_columns
                ),
                "id_like_columns": (
                    id_like_columns
                ),
                "split_like_columns": (
                    split_like_columns
                ),
                "sample_records": (
                    sample
                    .astype(object)
                    .where(
                        pd.notna(sample),
                        None,
                    )
                    .to_dict(
                        orient="records"
                    )
                ),
            }
        )

    except Exception as error:
        schema_records.append(
            {
                "relative_path": (
                    record["relative_path"]
                ),
                "status": "read_error",
                "size_bytes": (
                    path.stat().st_size
                ),
                "error": repr(error),
            }
        )

print(
    "CSV/TSV schemas inspected:",
    len(schema_records),
)

for record in schema_records:
    print()
    print("=" * 80)
    print(record["relative_path"])
    print("Status:", record["status"])

    if record["status"] != "readable":
        if "error" in record:
            print("Error:", record["error"])
        continue

    print("Columns:", record["columns"])
    print(
        "Label-like columns:",
        record["label_like_columns"],
    )
    print(
        "ID-like columns:",
        record["id_like_columns"],
    )
    print(
        "Split-like columns:",
        record["split_like_columns"],
    )

schema_report_path = (
    OUTPUT_DIR
    / "metadata_schema_report.json"
)

schema_report_path.write_text(
    json.dumps(
        schema_records,
        indent=2,
        sort_keys=True,
        default=str,
    )
    + "\n",
    encoding="utf-8",
)

print()
print("SAVED:", schema_report_path)
print()
print(
    "METADATA SCHEMA AUDIT COMPLETE"
)
print(
    "NO MODEL TRAINING OR TEST "
    "EVALUATION WAS PERFORMED"
)

In [ ]:
import json
from pathlib import Path

import pandas as pd

MINI_DIR = (
    IMAGE_DATASET / "data_mini"
)

MINI_METADATA_PATH = (
    MINI_DIR / "metadata_all.csv"
)

MINI_TRAIN_PATH = (
    MINI_DIR / "metadata_train.csv"
)

MINI_VAL_PATH = (
    MINI_DIR / "metadata_val.csv"
)

MINI_TEST_PATH = (
    MINI_DIR / "metadata_test.csv"
)

MINI_CLASS_MAP_PATH = (
    MINI_DIR / "class_map.json"
)

for path in [
    MINI_METADATA_PATH,
    MINI_TRAIN_PATH,
    MINI_VAL_PATH,
    MINI_TEST_PATH,
    MINI_CLASS_MAP_PATH,
]:
    assert path.is_file(), (
        f"Missing mini-cohort file: {path}"
    )

mini_all = pd.read_csv(
    MINI_METADATA_PATH
)

mini_train = pd.read_csv(
    MINI_TRAIN_PATH
)

mini_validation = pd.read_csv(
    MINI_VAL_PATH
)

mini_test = pd.read_csv(
    MINI_TEST_PATH
)

mini_class_map = json.loads(
    MINI_CLASS_MAP_PATH.read_text(
        encoding="utf-8"
    )
)

required_columns = {
    "gravityspy_id",
    "ml_label",
    "class_id",
    "split",
    "filename",
    "filepath",
}

for name, frame in [
    ("all", mini_all),
    ("train", mini_train),
    ("validation", mini_validation),
    ("test", mini_test),
]:
    missing = (
        required_columns
        - set(frame.columns)
    )

    assert not missing, (
        f"{name} is missing: {missing}"
    )

print("Mini class map:")
print(
    json.dumps(
        mini_class_map,
        indent=2,
        sort_keys=True,
    )
)

print()
print("--- Mini-cohort totals ---")
print("All rows:", len(mini_all))
print("Train rows:", len(mini_train))
print(
    "Validation rows:",
    len(mini_validation),
)
print("Test rows:", len(mini_test))
print(
    "Unique Gravity Spy IDs:",
    mini_all["gravityspy_id"].nunique(),
)
print(
    "Unique labels:",
    mini_all["ml_label"].nunique(),
)
print(
    "Duplicate Gravity Spy ID rows:",
    mini_all["gravityspy_id"]
    .duplicated(keep=False)
    .sum(),
)

print()
print("--- Split values ---")
print(
    mini_all["split"]
    .value_counts(dropna=False)
    .sort_index()
)

print()
print("--- Class counts by split ---")

mini_class_counts = (
    mini_all
    .groupby(
        ["ml_label", "split"],
        dropna=False,
    )
    .size()
    .unstack(
        fill_value=0
    )
    .sort_index()
)

display(mini_class_counts)

mini_summary = {
    "rows": int(len(mini_all)),
    "unique_gravityspy_ids": int(
        mini_all["gravityspy_id"].nunique()
    ),
    "classes": sorted(
        mini_all["ml_label"]
        .astype(str)
        .unique()
        .tolist()
    ),
    "class_count": int(
        mini_all["ml_label"].nunique()
    ),
    "split_counts": {
        str(key): int(value)
        for key, value in (
            mini_all["split"]
            .value_counts(
                dropna=False
            )
            .items()
        )
    },
    "class_split_counts": (
        mini_class_counts
        .reset_index()
        .to_dict(
            orient="records"
        )
    ),
}

mini_summary_path = (
    OUTPUT_DIR
    / "mini_cohort_summary.json"
)

mini_summary_path.write_text(
    json.dumps(
        mini_summary,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print()
print("SAVED:", mini_summary_path)
print()
print(
    "MINI IMAGE COHORT AUDIT COMPLETE"
)

In [ ]:
def resolve_image_path(
    row,
    mini_root,
):
    candidates = []

    filepath_value = row.get(
        "filepath"
    )

    filename_value = row.get(
        "filename"
    )

    if pd.notna(filepath_value):
        filepath = Path(
            str(filepath_value)
        )

        if filepath.is_absolute():
            candidates.append(filepath)
        else:
            candidates.extend(
                [
                    mini_root / filepath,
                    IMAGE_PROJECT / filepath,
                    mini_root
                    / "images"
                    / filepath.name,
                ]
            )

    if pd.notna(filename_value):
        filename = Path(
            str(filename_value)
        ).name

        candidates.extend(
            [
                mini_root
                / "images"
                / filename,
                mini_root / filename,
            ]
        )

    for candidate in candidates:
        if candidate.is_file():
            return str(
                candidate.resolve()
            )

    return None


mini_all[
    "resolved_image_path"
] = mini_all.apply(
    lambda row: resolve_image_path(
        row,
        MINI_DIR,
    ),
    axis=1,
)

available_mask = (
    mini_all[
        "resolved_image_path"
    ].notna()
)

print(
    "Metadata rows:",
    len(mini_all),
)
print(
    "Rows with local image:",
    int(available_mask.sum()),
)
print(
    "Rows without local image:",
    int((~available_mask).sum()),
)

print()
print(
    "Local-image availability by split:"
)

availability_by_split = (
    mini_all
    .assign(
        local_image_available=(
            available_mask
        )
    )
    .groupby("split")[
        "local_image_available"
    ]
    .agg(
        ["count", "sum"]
    )
)

availability_by_split[
    "missing"
] = (
    availability_by_split["count"]
    - availability_by_split["sum"]
)

display(availability_by_split)

missing_examples = (
    mini_all.loc[
        ~available_mask,
        [
            "gravityspy_id",
            "ml_label",
            "split",
            "filename",
            "filepath",
        ],
    ]
    .head(20)
)

if len(missing_examples):
    print()
    print(
        "First missing-image records:"
    )
    display(missing_examples)

availability_report = {
    "metadata_rows": int(
        len(mini_all)
    ),
    "local_images_available": int(
        available_mask.sum()
    ),
    "local_images_missing": int(
        (~available_mask).sum()
    ),
    "availability_by_split": (
        availability_by_split
        .reset_index()
        .to_dict(
            orient="records"
        )
    ),
}

availability_report_path = (
    OUTPUT_DIR
    / "mini_image_availability.json"
)

availability_report_path.write_text(
    json.dumps(
        availability_report,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print()
print(
    "SAVED:",
    availability_report_path,
)
print()
print(
    "MINI IMAGE AVAILABILITY "
    "AUDIT COMPLETE"
)

In [ ]:
TABULAR_JOIN_COLUMNS = [
    "source_record_id",
    "source_catalog",
    "ifo",
    "gravityspy_id",
    "peak_gps_ns",
    "ml_label",
    "random_split",
    "chronological_split",
    "duration",
    "peak_frequency",
    "central_freq",
    "bandwidth",
    "amplitude",
    "snr",
    "q_value",
]

tabular_join = pd.read_csv(
    PRIMARY_PATH,
    usecols=TABULAR_JOIN_COLUMNS,
    float_precision="round_trip",
)

print(
    "Frozen tabular rows:",
    len(tabular_join),
)
print(
    "Frozen unique Gravity Spy IDs:",
    tabular_join[
        "gravityspy_id"
    ].nunique(),
)
print(
    "Tabular rows in duplicated "
    "Gravity Spy IDs:",
    int(
        tabular_join[
            "gravityspy_id"
        ]
        .duplicated(
            keep=False
        )
        .sum()
    ),
)

mini_identity = (
    mini_all[
        [
            "gravityspy_id",
            "ml_label",
            "split",
            "filename",
            "resolved_image_path",
        ]
    ]
    .rename(
        columns={
            "ml_label": (
                "image_ml_label"
            ),
            "split": (
                "image_split"
            ),
        }
    )
)

joined = mini_identity.merge(
    tabular_join,
    on="gravityspy_id",
    how="left",
    indicator=True,
    validate="many_to_many",
)

join_counts = (
    joined["_merge"]
    .value_counts(
        dropna=False
    )
    .to_dict()
)

matched = joined.loc[
    joined["_merge"] == "both"
].copy()

matched[
    "label_matches"
] = (
    matched["image_ml_label"]
    .astype(str)
    .eq(
        matched["ml_label"]
        .astype(str)
    )
)

print()
print("Join counts:", join_counts)
print(
    "Matched joined rows:",
    len(matched),
)
print(
    "Matched unique image IDs:",
    matched[
        "gravityspy_id"
    ].nunique(),
)
print(
    "Unmatched unique image IDs:",
    joined.loc[
        joined["_merge"]
        == "left_only",
        "gravityspy_id",
    ].nunique(),
)
print(
    "Label-consistent matched rows:",
    int(
        matched[
            "label_matches"
        ].sum()
    ),
)
print(
    "Label-inconsistent matched rows:",
    int(
        (~matched[
            "label_matches"
        ]).sum()
    ),
)

print()
print(
    "Matched image classes:",
    sorted(
        matched[
            "image_ml_label"
        ]
        .astype(str)
        .unique()
        .tolist()
    ),
)

print()
print(
    "Frozen random-split membership "
    "among matched records:"
)
print(
    matched[
        "random_split"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)

print()
print(
    "Frozen chronological-split "
    "membership among matched records:"
)
print(
    matched[
        "chronological_split"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)

join_report = {
    "mini_rows": int(
        len(mini_all)
    ),
    "mini_unique_gravityspy_ids": int(
        mini_all[
            "gravityspy_id"
        ].nunique()
    ),
    "join_counts": {
        str(key): int(value)
        for key, value in (
            join_counts.items()
        )
    },
    "matched_join_rows": int(
        len(matched)
    ),
    "matched_unique_image_ids": int(
        matched[
            "gravityspy_id"
        ].nunique()
    ),
    "unmatched_unique_image_ids": int(
        joined.loc[
            joined["_merge"]
            == "left_only",
            "gravityspy_id",
        ].nunique()
    ),
    "label_consistent_rows": int(
        matched[
            "label_matches"
        ].sum()
    ),
    "label_inconsistent_rows": int(
        (~matched[
            "label_matches"
        ]).sum()
    ),
    "matched_classes": sorted(
        matched[
            "image_ml_label"
        ]
        .astype(str)
        .unique()
        .tolist()
    ),
    "random_split_counts": {
        str(key): int(value)
        for key, value in (
            matched[
                "random_split"
            ]
            .value_counts(
                dropna=False
            )
            .items()
        )
    },
    "chronological_split_counts": {
        str(key): int(value)
        for key, value in (
            matched[
                "chronological_split"
            ]
            .value_counts(
                dropna=False
            )
            .items()
        )
    },
}

join_report_path = (
    OUTPUT_DIR
    / "mini_to_frozen_tabular_join.json"
)

join_report_path.write_text(
    json.dumps(
        join_report,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

joined_output_path = (
    OUTPUT_DIR
    / "mini_to_frozen_tabular_join.csv.gz"
)

joined.to_csv(
    joined_output_path,
    index=False,
    compression="gzip",
)

print()
print("SAVED:", join_report_path)
print("SAVED:", joined_output_path)
print()
print(
    "MINI-TO-TABULAR JOIN "
    "AUDIT COMPLETE"
)
print(
    "NO MODEL TRAINING OR TEST "
    "EVALUATION WAS PERFORMED"
)

In [ ]:
CNN_PREDICTIONS_PATH = (
    IMAGE_DATASET
    / "runs"
    / "exp_CNNBASE_v1_seed0"
    / "predictions_test.csv"
)

VIT_PREDICTIONS_PATH = (
    IMAGE_DATASET
    / "runs"
    / "exp_VITBASE_v1_seed0"
    / "predictions_test.csv"
)

for path in [
    CNN_PREDICTIONS_PATH,
    VIT_PREDICTIONS_PATH,
]:
    assert path.is_file(), (
        f"Missing prediction file: {path}"
    )

prediction_records = {}

for model, path in [
    ("CNN", CNN_PREDICTIONS_PATH),
    ("ViT", VIT_PREDICTIONS_PATH),
]:
    predictions = pd.read_csv(path)

    required = {
        "gravityspy_id",
        "y_true",
        "y_pred",
        "true_label",
        "pred_label",
        "p_max",
    }

    missing = (
        required
        - set(predictions.columns)
    )

    assert not missing, (
        f"{model} predictions missing "
        f"columns: {missing}"
    )

    duplicate_ids = int(
        predictions[
            "gravityspy_id"
        ]
        .duplicated(
            keep=False
        )
        .sum()
    )

    accuracy = float(
        (
            predictions["y_true"]
            == predictions["y_pred"]
        ).mean()
    )

    labels = sorted(
        predictions[
            "true_label"
        ]
        .astype(str)
        .unique()
        .tolist()
    )

    metadata_test_ids = set(
        mini_test[
            "gravityspy_id"
        ].astype(str)
    )

    prediction_ids = set(
        predictions[
            "gravityspy_id"
        ].astype(str)
    )

    record = {
        "path": str(path),
        "sha256": sha256_file(path),
        "rows": int(
            len(predictions)
        ),
        "unique_gravityspy_ids": int(
            predictions[
                "gravityspy_id"
            ].nunique()
        ),
        "duplicate_id_rows": (
            duplicate_ids
        ),
        "class_count": int(
            len(labels)
        ),
        "classes": labels,
        "reported_from_predictions_accuracy": (
            accuracy
        ),
        "prediction_ids_equal_metadata_test_ids": (
            prediction_ids
            == metadata_test_ids
        ),
        "prediction_ids_missing_from_metadata_test": (
            len(
                prediction_ids
                - metadata_test_ids
            )
        ),
        "metadata_test_ids_missing_predictions": (
            len(
                metadata_test_ids
                - prediction_ids
            )
        ),
    }

    prediction_records[model] = record

    print()
    print("=" * 80)
    print(model)
    print(
        json.dumps(
            record,
            indent=2,
        )
    )

prediction_audit_path = (
    OUTPUT_DIR
    / "archived_prediction_audit.json"
)

prediction_audit_path.write_text(
    json.dumps(
        prediction_records,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print()
print(
    "SAVED:",
    prediction_audit_path,
)
print()
print(
    "ARCHIVED IMAGE-PREDICTION "
    "AUDIT COMPLETE"
)

In [ ]:
FULL_IMAGE_DIR = (
    IMAGE_DATASET
    / "gravityspy_full_url1_v1_metaonly"
)

FULL_CLASS_MAP_PATH = (
    FULL_IMAGE_DIR / "class_map.json"
)

FULL_EXCLUDED_PATH = (
    FULL_IMAGE_DIR
    / "excluded_small_classes.csv"
)

FULL_SPLIT_SUMMARY_PATH = (
    FULL_IMAGE_DIR
    / "split_summary_by_class.csv"
)

for path in [
    FULL_CLASS_MAP_PATH,
    FULL_EXCLUDED_PATH,
    FULL_SPLIT_SUMMARY_PATH,
]:
    assert path.is_file(), (
        f"Missing full-cohort file: {path}"
    )

full_class_map = json.loads(
    FULL_CLASS_MAP_PATH.read_text(
        encoding="utf-8"
    )
)

excluded_classes = pd.read_csv(
    FULL_EXCLUDED_PATH
)

full_split_summary = pd.read_csv(
    FULL_SPLIT_SUMMARY_PATH
)

frozen_labels = sorted(
    tabular_join[
        "ml_label"
    ]
    .astype(str)
    .unique()
    .tolist()
)

image_labels = sorted(
    full_split_summary[
        "ml_label"
    ]
    .astype(str)
    .unique()
    .tolist()
)

missing_from_image = sorted(
    set(frozen_labels)
    - set(image_labels)
)

extra_in_image = sorted(
    set(image_labels)
    - set(frozen_labels)
)

print("Full image class map:")
print(
    json.dumps(
        full_class_map,
        indent=2,
        sort_keys=True,
    )
)

print()
print("Excluded classes:")
display(excluded_classes)

print()
print("Full image split summary:")
display(full_split_summary)

print()
print(
    "Frozen tabular class count:",
    len(frozen_labels),
)
print(
    "Full image class count:",
    len(image_labels),
)
print(
    "Classes missing from image cohort:",
    missing_from_image,
)
print(
    "Classes present only in image cohort:",
    extra_in_image,
)

label_space_report = {
    "frozen_tabular_class_count": (
        len(frozen_labels)
    ),
    "full_image_class_count": (
        len(image_labels)
    ),
    "frozen_tabular_classes": (
        frozen_labels
    ),
    "full_image_classes": (
        image_labels
    ),
    "missing_from_image": (
        missing_from_image
    ),
    "extra_in_image": (
        extra_in_image
    ),
    "exact_label_space_match": (
        frozen_labels == image_labels
    ),
    "excluded_class_records": (
        excluded_classes
        .astype(object)
        .where(
            pd.notna(excluded_classes),
            None,
        )
        .to_dict(
            orient="records"
        )
    ),
}

label_space_report_path = (
    OUTPUT_DIR
    / "full_image_label_space.json"
)

label_space_report_path.write_text(
    json.dumps(
        label_space_report,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print()
print(
    "SAVED:",
    label_space_report_path,
)
print()
print(
    "FULL IMAGE LABEL-SPACE "
    "AUDIT COMPLETE"
)

In [ ]:
from collections import Counter

FULL_METADATA_PATHS = {
    "train": (
        FULL_IMAGE_DIR
        / "metadata_train.csv"
    ),
    "val": (
        FULL_IMAGE_DIR
        / "metadata_val.csv"
    ),
    "test": (
        FULL_IMAGE_DIR
        / "metadata_test.csv"
    ),
}

for split_name, path in (
    FULL_METADATA_PATHS.items()
):
    assert path.is_file(), (
        f"Missing {split_name} metadata: "
        f"{path}"
    )

tabular_identity = (
    tabular_join[
        [
            "gravityspy_id",
            "ml_label",
            "ifo",
            "peak_gps_ns",
        ]
    ]
    .copy()
)

tabular_identity[
    "normalized_gravityspy_id"
] = (
    tabular_identity[
        "gravityspy_id"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)

tabular_id_set = set(
    tabular_identity[
        "normalized_gravityspy_id"
    ]
)

tabular_label_sets = (
    tabular_identity
    .groupby(
        "normalized_gravityspy_id"
    )["ml_label"]
    .agg(
        lambda values: set(
            values.astype(str)
        )
    )
    .to_dict()
)

audit_by_file = {}
overlap_examples = []
unmatched_examples = []

for split_name, path in (
    FULL_METADATA_PATHS.items()
):
    print()
    print("=" * 80)
    print(
        "Streaming:",
        split_name,
        path,
    )

    rows = 0
    unique_ids = set()
    overlap_ids = set()
    label_counts = Counter()
    era_counts = Counter()
    detector_counts = Counter()
    metadata_split_counts = Counter()
    nonempty_urls = 0
    nonempty_relative_paths = 0
    label_consistent_overlap_rows = 0
    label_inconsistent_overlap_rows = 0

    for chunk in pd.read_csv(
        path,
        usecols=[
            "gravityspy_id",
            "ml_label",
            "ifo",
            "era",
            "split",
            "url1",
            "relative_image_path",
        ],
        dtype={
            "gravityspy_id": "string",
            "ml_label": "string",
            "ifo": "string",
            "era": "string",
            "split": "string",
            "url1": "string",
            "relative_image_path": (
                "string"
            ),
        },
        chunksize=50_000,
    ):
        rows += len(chunk)

        normalized_ids = (
            chunk["gravityspy_id"]
            .astype("string")
            .str.strip()
            .str.lower()
        )

        unique_ids.update(
            normalized_ids.dropna()
        )

        overlap_mask = (
            normalized_ids.isin(
                tabular_id_set
            )
        )

        overlapping_chunk = (
            chunk.loc[
                overlap_mask,
                [
                    "gravityspy_id",
                    "ml_label",
                    "ifo",
                    "era",
                    "split",
                    "url1",
                    "relative_image_path",
                ],
            ]
            .copy()
        )

        overlapping_chunk[
            "normalized_gravityspy_id"
        ] = normalized_ids.loc[
            overlap_mask
        ].to_numpy()

        overlap_ids.update(
            overlapping_chunk[
                "normalized_gravityspy_id"
            ]
            .dropna()
        )

        for label, count in (
            chunk["ml_label"]
            .value_counts(
                dropna=False
            )
            .items()
        ):
            label_counts[
                str(label)
            ] += int(count)

        for era, count in (
            chunk["era"]
            .value_counts(
                dropna=False
            )
            .items()
        ):
            era_counts[
                str(era)
            ] += int(count)

        for detector, count in (
            chunk["ifo"]
            .value_counts(
                dropna=False
            )
            .items()
        ):
            detector_counts[
                str(detector)
            ] += int(count)

        for metadata_split, count in (
            chunk["split"]
            .value_counts(
                dropna=False
            )
            .items()
        ):
            metadata_split_counts[
                str(metadata_split)
            ] += int(count)

        nonempty_urls += int(
            chunk["url1"]
            .fillna("")
            .str.strip()
            .ne("")
            .sum()
        )

        nonempty_relative_paths += int(
            chunk[
                "relative_image_path"
            ]
            .fillna("")
            .str.strip()
            .ne("")
            .sum()
        )

        for row in (
            overlapping_chunk
            .itertuples(index=False)
        ):
            expected_labels = (
                tabular_label_sets[
                    row.normalized_gravityspy_id
                ]
            )

            if str(row.ml_label) in (
                expected_labels
            ):
                label_consistent_overlap_rows += 1
            else:
                label_inconsistent_overlap_rows += 1

            if (
                len(overlap_examples)
                < 20
            ):
                overlap_examples.append(
                    {
                        "metadata_file": (
                            split_name
                        ),
                        "gravityspy_id": (
                            str(
                                row.gravityspy_id
                            )
                        ),
                        "image_ml_label": (
                            str(row.ml_label)
                        ),
                        "tabular_labels": (
                            sorted(
                                expected_labels
                            )
                        ),
                        "ifo": str(row.ifo),
                        "era": str(row.era),
                        "split": (
                            str(row.split)
                        ),
                        "relative_image_path": (
                            str(
                                row.relative_image_path
                            )
                        ),
                    }
                )

        if len(unmatched_examples) < 20:
            unmatched_rows = (
                chunk.loc[
                    ~overlap_mask,
                    [
                        "gravityspy_id",
                        "ml_label",
                        "era",
                    ],
                ]
                .head(
                    20
                    - len(
                        unmatched_examples
                    )
                )
            )

            unmatched_examples.extend(
                unmatched_rows
                .astype(object)
                .where(
                    pd.notna(
                        unmatched_rows
                    ),
                    None,
                )
                .to_dict(
                    orient="records"
                )
            )

    record = {
        "path": str(path),
        "sha256": sha256_file(path),
        "rows": int(rows),
        "unique_gravityspy_ids": int(
            len(unique_ids)
        ),
        "overlap_unique_ids": int(
            len(overlap_ids)
        ),
        "overlap_fraction_of_file": (
            len(overlap_ids)
            / len(unique_ids)
            if unique_ids
            else 0.0
        ),
        "label_consistent_overlap_rows": int(
            label_consistent_overlap_rows
        ),
        "label_inconsistent_overlap_rows": int(
            label_inconsistent_overlap_rows
        ),
        "label_counts": dict(
            sorted(
                label_counts.items()
            )
        ),
        "era_counts": dict(
            sorted(
                era_counts.items()
            )
        ),
        "detector_counts": dict(
            sorted(
                detector_counts.items()
            )
        ),
        "metadata_split_counts": dict(
            sorted(
                metadata_split_counts.items()
            )
        ),
        "nonempty_url1_rows": int(
            nonempty_urls
        ),
        "nonempty_relative_image_path_rows": int(
            nonempty_relative_paths
        ),
    }

    audit_by_file[
        split_name
    ] = record

    print(
        json.dumps(
            record,
            indent=2,
        )
    )

full_overlap_report = {
    "frozen_tabular_rows": int(
        len(tabular_join)
    ),
    "frozen_unique_gravityspy_ids": int(
        tabular_join[
            "gravityspy_id"
        ].nunique()
    ),
    "metadata_files": audit_by_file,
    "overlap_examples": (
        overlap_examples
    ),
    "unmatched_examples": (
        unmatched_examples
    ),
    "training_performed": False,
    "test_evaluated": False,
}

full_overlap_report_path = (
    OUTPUT_DIR
    / "full_image_to_frozen_overlap.json"
)

full_overlap_report_path.write_text(
    json.dumps(
        full_overlap_report,
        indent=2,
        sort_keys=True,
        default=str,
    )
    + "\n",
    encoding="utf-8",
)

print()
print(
    "SAVED:",
    full_overlap_report_path,
)
print()
print(
    "FULL IMAGE-TO-TABULAR "
    "OVERLAP AUDIT COMPLETE"
)

In [ ]:
local_path_audit = {}

for split_name, path in (
    FULL_METADATA_PATHS.items()
):
    sample = pd.read_csv(
        path,
        usecols=[
            "gravityspy_id",
            "url1",
            "relative_image_path",
        ],
        dtype="string",
        nrows=5000,
    )

    available = 0
    candidate_examples = []

    for row in sample.itertuples(
        index=False
    ):
        relative_value = (
            ""
            if pd.isna(
                row.relative_image_path
            )
            else str(
                row.relative_image_path
            ).strip()
        )

        candidates = []

        if relative_value:
            relative_path = Path(
                relative_value
            )

            if relative_path.is_absolute():
                candidates.append(
                    relative_path
                )
            else:
                candidates.extend(
                    [
                        FULL_IMAGE_DIR
                        / relative_path,
                        IMAGE_DATASET
                        / relative_path,
                        IMAGE_PROJECT
                        / relative_path,
                    ]
                )

        existing = [
            candidate
            for candidate in candidates
            if candidate.is_file()
        ]

        if existing:
            available += 1

        if (
            len(candidate_examples) < 10
        ):
            candidate_examples.append(
                {
                    "gravityspy_id": (
                        str(
                            row.gravityspy_id
                        )
                    ),
                    "relative_image_path": (
                        relative_value
                    ),
                    "url1_present": bool(
                        pd.notna(row.url1)
                        and str(
                            row.url1
                        ).strip()
                    ),
                    "local_file_found": bool(
                        existing
                    ),
                    "resolved_path": (
                        str(
                            existing[0]
                        )
                        if existing
                        else None
                    ),
                }
            )

    local_path_audit[
        split_name
    ] = {
        "sample_rows": int(
            len(sample)
        ),
        "local_files_found": int(
            available
        ),
        "local_availability_fraction": (
            available / len(sample)
            if len(sample)
            else 0.0
        ),
        "examples": (
            candidate_examples
        ),
    }

print(
    json.dumps(
        local_path_audit,
        indent=2,
    )
)

local_path_report_path = (
    OUTPUT_DIR
    / "full_image_local_availability.json"
)

local_path_report_path.write_text(
    json.dumps(
        local_path_audit,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print()
print(
    "SAVED:",
    local_path_report_path,
)
print()
print(
    "FULL IMAGE LOCAL-AVAILABILITY "
    "AUDIT COMPLETE"
)
print(
    "NO IMAGES WERE DOWNLOADED"
)

In [ ]:
full_assignment_parts = []

for image_split_name, path in (
    FULL_METADATA_PATHS.items()
):
    print(
        "Loading assignments:",
        image_split_name,
    )

    for chunk in pd.read_csv(
        path,
        usecols=[
            "gravityspy_id",
            "ml_label",
            "era",
            "split",
        ],
        dtype="string",
        chunksize=100_000,
    ):
        chunk = chunk.copy()

        chunk[
            "normalized_gravityspy_id"
        ] = (
            chunk["gravityspy_id"]
            .str.strip()
            .str.lower()
        )

        chunk[
            "source_metadata_file"
        ] = image_split_name

        full_assignment_parts.append(
            chunk[
                [
                    "normalized_gravityspy_id",
                    "gravityspy_id",
                    "ml_label",
                    "era",
                    "split",
                    "source_metadata_file",
                ]
            ]
        )

full_assignments = pd.concat(
    full_assignment_parts,
    ignore_index=True,
)

del full_assignment_parts

print()
print(
    "Full image metadata rows:",
    len(full_assignments),
)
print(
    "Unique image metadata IDs:",
    full_assignments[
        "normalized_gravityspy_id"
    ].nunique(),
)
print(
    "Rows in duplicated IDs:",
    int(
        full_assignments[
            "normalized_gravityspy_id"
        ]
        .duplicated(
            keep=False
        )
        .sum()
    ),
)

duplicate_consistency = (
    full_assignments
    .groupby(
        "normalized_gravityspy_id"
    )
    .agg(
        split_count=(
            "split",
            "nunique",
        ),
        label_count=(
            "ml_label",
            "nunique",
        ),
        era_count=(
            "era",
            "nunique",
        ),
    )
)

conflicting_duplicates = (
    duplicate_consistency.loc[
        (
            duplicate_consistency[
                "split_count"
            ]
            > 1
        )
        | (
            duplicate_consistency[
                "label_count"
            ]
            > 1
        )
        | (
            duplicate_consistency[
                "era_count"
            ]
            > 1
        )
    ]
)

print(
    "Conflicting duplicated IDs:",
    len(conflicting_duplicates),
)

assert len(
    conflicting_duplicates
) == 0, (
    "Image metadata contains conflicting "
    "assignments for duplicated IDs."
)

image_assignments_unique = (
    full_assignments
    .drop_duplicates(
        subset=[
            "normalized_gravityspy_id"
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

tabular_split_assignments = (
    tabular_join[
        [
            "gravityspy_id",
            "ml_label",
            "random_split",
            "chronological_split",
        ]
    ]
    .copy()
)

tabular_split_assignments[
    "normalized_gravityspy_id"
] = (
    tabular_split_assignments[
        "gravityspy_id"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)

tabular_split_assignments = (
    tabular_split_assignments
    .drop_duplicates(
        subset=[
            "normalized_gravityspy_id"
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

split_alignment = (
    tabular_split_assignments
    .merge(
        image_assignments_unique[
            [
                "normalized_gravityspy_id",
                "ml_label",
                "era",
                "split",
            ]
        ].rename(
            columns={
                "ml_label": (
                    "image_ml_label"
                ),
                "split": (
                    "image_split"
                ),
            }
        ),
        on=(
            "normalized_gravityspy_id"
        ),
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)

split_alignment[
    "label_matches"
] = (
    split_alignment["ml_label"]
    .astype(str)
    .eq(
        split_alignment[
            "image_ml_label"
        ].astype(str)
    )
)

print()
print("Join status:")
print(
    split_alignment["_merge"]
    .value_counts(
        dropna=False
    )
)

print()
print(
    "Label consistency:"
)
print(
    split_alignment[
        "label_matches"
    ]
    .value_counts(
        dropna=False
    )
)

random_crosstab = pd.crosstab(
    split_alignment[
        "random_split"
    ],
    split_alignment[
        "image_split"
    ],
    margins=True,
)

chronological_crosstab = (
    pd.crosstab(
        split_alignment[
            "chronological_split"
        ],
        split_alignment[
            "image_split"
        ],
        margins=True,
    )
)

print()
print(
    "Image split versus frozen "
    "grouped-random split:"
)
display(random_crosstab)

print()
print(
    "Image split versus frozen "
    "chronological split:"
)
display(chronological_crosstab)

random_test_seen_by_image_training = int(
    (
        split_alignment[
            "random_split"
        ].eq("test")
        & split_alignment[
            "image_split"
        ].eq("train")
    ).sum()
)

chronological_test_seen_by_image_training = int(
    (
        split_alignment[
            "chronological_split"
        ].eq("test")
        & split_alignment[
            "image_split"
        ].eq("train")
    ).sum()
)

random_validation_seen_by_image_training = int(
    (
        split_alignment[
            "random_split"
        ].eq("validation")
        & split_alignment[
            "image_split"
        ].eq("train")
    ).sum()
)

chronological_validation_seen_by_image_training = int(
    (
        split_alignment[
            "chronological_split"
        ].eq("validation")
        & split_alignment[
            "image_split"
        ].eq("train")
    ).sum()
)

print()
print(
    "Frozen grouped-random test events "
    "inside image training split:",
    random_test_seen_by_image_training,
)
print(
    "Frozen chronological test events "
    "inside image training split:",
    chronological_test_seen_by_image_training,
)
print(
    "Frozen grouped-random validation "
    "events inside image training split:",
    random_validation_seen_by_image_training,
)
print(
    "Frozen chronological validation "
    "events inside image training split:",
    chronological_validation_seen_by_image_training,
)

split_alignment_report = {
    "frozen_unique_ids": int(
        len(
            tabular_split_assignments
        )
    ),
    "matched_unique_ids": int(
        split_alignment["_merge"]
        .eq("both")
        .sum()
    ),
    "unmatched_unique_ids": int(
        split_alignment["_merge"]
        .eq("left_only")
        .sum()
    ),
    "label_consistent_ids": int(
        split_alignment[
            "label_matches"
        ].sum()
    ),
    "label_inconsistent_ids": int(
        (~split_alignment[
            "label_matches"
        ]).sum()
    ),
    "random_split_crosstab": (
        random_crosstab
        .to_dict()
    ),
    "chronological_split_crosstab": (
        chronological_crosstab
        .to_dict()
    ),
    "frozen_random_test_in_image_train": (
        random_test_seen_by_image_training
    ),
    "frozen_chronological_test_in_image_train": (
        chronological_test_seen_by_image_training
    ),
    "frozen_random_validation_in_image_train": (
        random_validation_seen_by_image_training
    ),
    "frozen_chronological_validation_in_image_train": (
        chronological_validation_seen_by_image_training
    ),
    "archived_image_split_matches_frozen_random": (
        bool(
            (
                split_alignment[
                    "image_split"
                ]
                == split_alignment[
                    "random_split"
                ]
            ).all()
        )
    ),
    "archived_image_split_matches_frozen_chronological": (
        bool(
            (
                split_alignment[
                    "image_split"
                ]
                == split_alignment[
                    "chronological_split"
                ]
            ).all()
        )
    ),
}

split_alignment_path = (
    OUTPUT_DIR
    / "image_tabular_split_alignment.json"
)

split_alignment_path.write_text(
    json.dumps(
        split_alignment_report,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

split_alignment_csv_path = (
    OUTPUT_DIR
    / "image_tabular_split_alignment.csv.gz"
)

split_alignment.to_csv(
    split_alignment_csv_path,
    index=False,
    compression="gzip",
)

print()
print("SAVED:", split_alignment_path)
print(
    "SAVED:",
    split_alignment_csv_path,
)
print()
print(
    "IMAGE–TABULAR SPLIT-ALIGNMENT "
    "AUDIT COMPLETE"
)

In [ ]:
random_split_aligned = (
    split_alignment_report[
        "archived_image_split_matches_frozen_random"
    ]
)

chronological_split_aligned = (
    split_alignment_report[
        "archived_image_split_matches_frozen_chronological"
    ]
)

all_ids_mapped = (
    split_alignment_report[
        "matched_unique_ids"
    ]
    == split_alignment_report[
        "frozen_unique_ids"
    ]
)

all_labels_consistent = (
    split_alignment_report[
        "label_inconsistent_ids"
    ]
    == 0
)

local_full_images_available = any(
    record[
        "local_files_found"
    ]
    > 0
    for record in (
        local_path_audit.values()
    )
)

mini_is_matched = (
    join_report[
        "matched_unique_image_ids"
    ]
    > 0
    and prediction_records[
        "CNN"
    ][
        "class_count"
    ]
    == 24
)

matched_existing_baseline_feasible = (
    all_ids_mapped
    and all_labels_consistent
    and local_full_images_available
    and (
        random_split_aligned
        or chronological_split_aligned
    )
)

decision = {
    "status": (
        "matched_existing_image_baseline_not_feasible"
        if not matched_existing_baseline_feasible
        else "matched_existing_image_baseline_feasible"
    ),
    "gates": {
        "same_24_class_label_space": (
            label_space_report[
                "exact_label_space_match"
            ]
        ),
        "all_frozen_ids_present_in_full_metadata": (
            all_ids_mapped
        ),
        "overlap_labels_consistent": (
            all_labels_consistent
        ),
        "archived_split_matches_frozen_random": (
            random_split_aligned
        ),
        "archived_split_matches_frozen_chronological": (
            chronological_split_aligned
        ),
        "full_images_available_locally": (
            local_full_images_available
        ),
        "mini_predictions_form_matched_baseline": (
            mini_is_matched
        ),
    },
    "mini_experiment": {
        "classes": int(
            mini_summary[
                "class_count"
            ]
        ),
        "rows": int(
            mini_summary["rows"]
        ),
        "frozen_cohort_overlap_ids": int(
            join_report[
                "matched_unique_image_ids"
            ]
        ),
        "archived_models": [
            "CNN",
            "ViT",
        ],
        "seeds": 1,
    },
    "full_metadata_experiment": {
        "classes": int(
            label_space_report[
                "full_image_class_count"
            ]
        ),
        "frozen_ids_mapped": int(
            split_alignment_report[
                "matched_unique_ids"
            ]
        ),
        "local_image_sample_availability": (
            0.0
        ),
        "urls_present": True,
    },
    "why_existing_results_are_not_comparable": [
        (
            "The archived mini image experiment "
            "contains only 10 classes and has "
            "zero event overlap with the frozen "
            "tabular O3 cohort."
        ),
        (
            "The full image metadata covers the "
            "24-class frozen cohort, but the image "
            "files are not stored locally."
        ),
        (
            "The archived image train/validation/"
            "test assignments differ from both "
            "frozen tabular split protocols."
        ),
        (
            "Using a model trained with the "
            "archived image split would expose "
            "part of the frozen tabular test set "
            "during image-model training."
        ),
        (
            "A controlled comparison would "
            "therefore require downloading the "
            "O3 images and retraining image "
            "models from scratch under the frozen "
            "grouped-random and chronological "
            "assignments."
        ),
    ],
    "revision_action": (
        "Do not report the archived CNN or ViT "
        "numbers as a numerical baseline. Add a "
        "transparent limitation explaining why "
        "a controlled image-versus-tabular "
        "comparison was not performed, and "
        "contextualize the distinction using "
        "published image-based Gravity Spy work."
    ),
    "training_performed": False,
    "images_downloaded": False,
    "test_evaluated": False,
}

decision_path = (
    OUTPUT_DIR
    / "image_comparison_feasibility_decision.json"
)

decision_path.write_text(
    json.dumps(
        decision,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print(
    json.dumps(
        decision,
        indent=2,
    )
)

print()
print("SAVED:", decision_path)
print()
print(
    "IMAGE-COMPARISON FEASIBILITY "
    "DECISION COMPLETE"
)

In [ ]:
from pathlib import Path
import json
import hashlib
import pandas as pd
from datetime import datetime, timezone

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
)

FREEZE_DIR = ROOT / "revision_evidence_freeze_v1"
FREEZE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Revision root:", ROOT)
print("Freeze directory:", FREEZE_DIR)

In [ ]:
EVIDENCE = {
    "frozen_dataset_manifest": (
        ROOT
        / "jpco_o3_splits_v1"
        / "manifest.json"
    ),

    "frozen_split_summary": (
        ROOT
        / "jpco_o3_splits_v1"
        / "split_summary.csv"
    ),

    "xgboost_temporal_results": (
        ROOT
        / "xgboost_stage2_results"
        / "test_summary.csv"
    ),

    "mlp_temporal_results": (
        ROOT
        / "mlp_stage2_results"
        / "test_summary.csv"
    ),

    "fttransformer_temporal_results": (
        ROOT
        / "fttransformer_stage2"
        / "test_summary.csv"
    ),

    "targeted_ablation_summary": (
        ROOT
        / "targeted_ablation_v1"
        / "test_summary.csv"
    ),

    "permutation_importance_summary": (
        ROOT
        / "common_permutation_importance_v1"
        / "summary.json"
    ),

    "complexity_summary": (
        ROOT
        / "complexity_benchmark_v1"
        / "complexity_summary.csv"
    ),

    "latency_summary": (
        ROOT
        / "complexity_benchmark_v1"
        / "latency_summary.csv"
    ),

    "feature_scope_report": (
        ROOT
        / "feature_provenance_v1"
        / "canonical_scope_report.json"
    ),

    "historical_sampling_audit": (
        ROOT
        / "feature_provenance_v1"
        / "historical_sampling_final_audit.json"
    ),

    "bandwidth_summary": (
        ROOT
        / "bandwidth_augmentation_v1"
        / "full_suite"
        / "test_summary.csv"
    ),

    "bandwidth_paired_differences": (
        ROOT
        / "bandwidth_augmentation_v1"
        / "full_suite"
        / "paired_difference_summary.csv"
    ),

    "image_feasibility_decision": (
        ROOT
        / "image_comparison_audit_v1"
        / "image_comparison_feasibility_decision.json"
    ),

    "image_split_alignment": (
        ROOT
        / "image_comparison_audit_v1"
        / "image_tabular_split_alignment.json"
    ),
}

for name, path in EVIDENCE.items():
    status = "FOUND" if path.is_file() else "MISSING"
    print(f"{status}: {name}")
    print(" ", path)

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
)

print("Searching for targeted-ablation summaries...")
print()

ablation_candidates = sorted(
    path
    for path in ROOT.rglob("test_summary.csv")
    if (
        "targeted_ablation" in str(path).lower()
        or "ablation" in str(path).lower()
    )
)

print(
    "Targeted-ablation candidates:",
    len(ablation_candidates),
)

for index, path in enumerate(
    ablation_candidates,
    start=1,
):
    print(f"\n{index}. {path}")
    print(
        "   bytes:",
        f"{path.stat().st_size:,}",
    )

    try:
        frame = pd.read_csv(path)

        print(
            "   rows:",
            len(frame),
        )
        print(
            "   columns:",
            frame.columns.tolist(),
        )

        if len(frame) > 0:
            print(
                frame.head(3).to_string(
                    index=False
                )
            )

    except Exception as error:
        print(
            "   READ ERROR:",
            repr(error),
        )


print()
print("=" * 80)
print("Searching for permutation-importance summaries...")
print()

permutation_candidates = sorted(
    {
        path
        for pattern in [
            "*summary*.json",
            "*summary*.csv",
            "*correlation*.csv",
            "*importance*.csv",
        ]
        for path in ROOT.rglob(pattern)
        if (
            "permutation" in str(path).lower()
            or "common_importance" in str(path).lower()
        )
    }
)

print(
    "Permutation-importance candidates:",
    len(permutation_candidates),
)

for index, path in enumerate(
    permutation_candidates,
    start=1,
):
    print(f"\n{index}. {path}")
    print(
        "   bytes:",
        f"{path.stat().st_size:,}",
    )

    try:
        if path.suffix.lower() == ".json":
            with path.open() as handle:
                content = json.load(handle)

            if isinstance(content, dict):
                print(
                    "   top-level keys:",
                    list(content.keys()),
                )
            else:
                print(
                    "   JSON type:",
                    type(content).__name__,
                )

        elif path.suffix.lower() == ".csv":
            frame = pd.read_csv(path)

            print(
                "   rows:",
                len(frame),
            )
            print(
                "   columns:",
                frame.columns.tolist(),
            )

            if len(frame) > 0:
                print(
                    frame.head(3).to_string(
                        index=False
                    )
                )

    except Exception as error:
        print(
            "   READ ERROR:",
            repr(error),
        )

print()
print("MISSING-EVIDENCE SEARCH COMPLETE")
print("NO FILES WERE MODIFIED")

In [ ]:
from pathlib import Path
import pandas as pd
import json

PERM_DIR = (
    ROOT
    / "permutation_importance_v1"
)

assert PERM_DIR.is_dir(), (
    f"Missing directory: {PERM_DIR}"
)

files = sorted(
    path
    for path in PERM_DIR.rglob("*")
    if path.is_file()
)

print(
    "Permutation package files:",
    len(files),
)

for index, path in enumerate(
    files,
    start=1,
):
    relative = path.relative_to(PERM_DIR)

    print(
        f"{index:>2}. {relative} "
        f"({path.stat().st_size:,} bytes)"
    )

print()
print("PERMUTATION PACKAGE INVENTORY COMPLETE")

In [ ]:
from pathlib import Path
import json
import hashlib
import pandas as pd
from datetime import datetime, timezone

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
)

FREEZE_DIR = (
    ROOT
    / "revision_evidence_freeze_v1"
)

FREEZE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EVIDENCE = {
    # Frozen dataset and split construction
    "frozen_dataset_manifest": (
        ROOT
        / "jpco_o3_splits_v1"
        / "manifest.json"
    ),

    "frozen_split_summary": (
        ROOT
        / "jpco_o3_splits_v1"
        / "split_summary.csv"
    ),

    # Temporal generalization and timestamp removal
    "xgboost_temporal_results": (
        ROOT
        / "xgboost_stage2_results"
        / "test_summary.csv"
    ),

    "mlp_temporal_results": (
        ROOT
        / "mlp_stage2_results"
        / "test_summary.csv"
    ),

    "fttransformer_temporal_results": (
        ROOT
        / "fttransformer_stage2"
        / "test_summary.csv"
    ),

    # Targeted feature ablations
    "targeted_ablation_summary": (
        ROOT
        / "targeted_ablation_v1"
        / "full_suite"
        / "test_summary.csv"
    ),

    # Common-method permutation importance
    "permutation_protocol": (
        ROOT
        / "permutation_importance_v1"
        / "protocol.json"
    ),

    "permutation_baseline_summary": (
        ROOT
        / "permutation_importance_v1"
        / "baseline_summary.csv"
    ),

    "permutation_importance_per_seed": (
        ROOT
        / "permutation_importance_v1"
        / "permutation_importance_per_seed.csv"
    ),

    "permutation_importance_summary": (
        ROOT
        / "permutation_importance_v1"
        / "permutation_importance_summary.csv"
    ),

    "permutation_rank_agreement": (
        ROOT
        / "permutation_importance_v1"
        / "rank_agreement.csv"
    ),

    # Operational complexity
    "complexity_summary": (
        ROOT
        / "complexity_benchmark_v1"
        / "complexity_summary.csv"
    ),

    "latency_summary": (
        ROOT
        / "complexity_benchmark_v1"
        / "latency_summary.csv"
    ),

    # Feature provenance
    "feature_scope_report": (
        ROOT
        / "feature_provenance_v1"
        / "canonical_scope_report.json"
    ),

    "historical_sampling_audit": (
        ROOT
        / "feature_provenance_v1"
        / "historical_sampling_final_audit.json"
    ),

    # Richer-feature experiment
    "bandwidth_summary": (
        ROOT
        / "bandwidth_augmentation_v1"
        / "full_suite"
        / "test_summary.csv"
    ),

    "bandwidth_paired_differences": (
        ROOT
        / "bandwidth_augmentation_v1"
        / "full_suite"
        / "paired_difference_summary.csv"
    ),

    # Image-comparison feasibility
    "image_feasibility_decision": (
        ROOT
        / "image_comparison_audit_v1"
        / "image_comparison_feasibility_decision.json"
    ),

    "image_split_alignment": (
        ROOT
        / "image_comparison_audit_v1"
        / "image_tabular_split_alignment.json"
    ),
}

print("--- Corrected authoritative evidence inventory ---")
print()

missing = []

for name, path in EVIDENCE.items():
    exists = path.is_file()

    print(
        f"{'FOUND' if exists else 'MISSING'}: "
        f"{name}"
    )
    print(" ", path)

    if not exists:
        missing.append(name)

print()
print("Evidence entries:", len(EVIDENCE))
print("Missing entries:", len(missing))

assert len(EVIDENCE) == 19
assert not missing, (
    "Missing evidence entries: "
    + ", ".join(missing)
)

print()
print("ALL AUTHORITATIVE EVIDENCE FILES LOCATED")

In [ ]:
# Dataset and split checks
with EVIDENCE[
    "frozen_dataset_manifest"
].open() as handle:
    manifest = json.load(handle)

assert manifest["primary_rows"] == 437753
assert manifest["primary_groups"] == 437750
assert manifest["target"] == "ml_label"

split_summary = pd.read_csv(
    EVIDENCE["frozen_split_summary"]
)

assert set(
    split_summary["configuration"]
) == {
    "random_split",
    "chronological_split",
}

assert set(
    split_summary["partition"]
) == {
    "train",
    "validation",
    "test",
}


# Temporal-result checks
for evidence_name in [
    "xgboost_temporal_results",
    "mlp_temporal_results",
    "fttransformer_temporal_results",
]:
    frame = pd.read_csv(
        EVIDENCE[evidence_name]
    )

    assert len(frame) == 4, (
        f"Unexpected rows in {evidence_name}: "
        f"{len(frame)}"
    )

    assert set(
        frame["condition"]
    ) == {
        "random_split__original_nine",
        "random_split__timestamp_free_six",
        "chronological_split__original_nine",
        "chronological_split__timestamp_free_six",
    }

    assert (
        frame["seeds"] == 3
    ).all()


# Targeted-ablation checks
targeted = pd.read_csv(
    EVIDENCE[
        "targeted_ablation_summary"
    ]
)

assert len(targeted) == 8

assert set(
    targeted["model"]
) == {
    "XGBoost",
    "MLP",
}

assert set(
    targeted["condition"]
) == {
    "remove_start_time_ns",
    "remove_peak_time_pair",
    "timestamp_free_remove_amplitude_snr",
    "timestamp_free_remove_q",
}

assert (
    targeted["seeds"] == 3
).all()


# Permutation-importance checks
permutation_baseline = pd.read_csv(
    EVIDENCE[
        "permutation_baseline_summary"
    ]
)

permutation_per_seed = pd.read_csv(
    EVIDENCE[
        "permutation_importance_per_seed"
    ]
)

permutation_summary = pd.read_csv(
    EVIDENCE[
        "permutation_importance_summary"
    ]
)

rank_agreement = pd.read_csv(
    EVIDENCE[
        "permutation_rank_agreement"
    ]
)

with EVIDENCE[
    "permutation_protocol"
].open() as handle:
    permutation_protocol = json.load(handle)

assert len(permutation_baseline) == 36
assert len(permutation_per_seed) == 540
assert len(permutation_summary) == 180
assert len(rank_agreement) > 0

assert set(
    permutation_per_seed["model"]
) == {
    "XGBoost",
    "MLP",
    "FT-Transformer",
}

assert set(
    permutation_per_seed["seed"]
) == {
    42,
    43,
    44,
}

assert (
    permutation_baseline[
        "restored_full_test_predictions_verified"
    ]
    .astype(str)
    .str.lower()
    .eq("true")
    .all()
)


# Operational-complexity checks
complexity = pd.read_csv(
    EVIDENCE["complexity_summary"]
)

latency = pd.read_csv(
    EVIDENCE["latency_summary"]
)

assert len(complexity) == 3
assert len(latency) == 12

assert set(
    complexity["model"]
) == {
    "XGBoost",
    "MLP",
    "FT-Transformer",
}


# Feature-provenance checks
with EVIDENCE[
    "feature_scope_report"
].open() as handle:
    feature_scope = json.load(handle)

with EVIDENCE[
    "historical_sampling_audit"
].open() as handle:
    historical_sampling = json.load(
        handle
    )

assert (
    historical_sampling["status"]
    == "resolved_at_class_count_level"
)

assert (
    historical_sampling[
        "historical_final_sample_rows"
    ]
    == 50554
)


# Bandwidth-augmentation checks
bandwidth = pd.read_csv(
    EVIDENCE["bandwidth_summary"]
)

bandwidth_differences = pd.read_csv(
    EVIDENCE[
        "bandwidth_paired_differences"
    ]
)

assert len(bandwidth) == 4
assert len(bandwidth_differences) == 4

assert set(
    bandwidth["model"]
) == {
    "XGBoost",
    "MLP",
}

assert set(
    bandwidth["split"]
) == {
    "random_split",
    "chronological_split",
}

assert (
    bandwidth["seeds"] == 3
).all()


# Image-feasibility checks
with EVIDENCE[
    "image_feasibility_decision"
].open() as handle:
    image_decision = json.load(handle)

with EVIDENCE[
    "image_split_alignment"
].open() as handle:
    image_alignment = json.load(handle)

assert (
    image_decision["status"]
    == "matched_existing_image_baseline_not_feasible"
)

assert (
    image_decision[
        "training_performed"
    ]
    is False
)

assert (
    image_decision[
        "images_downloaded"
    ]
    is False
)

assert (
    image_decision[
        "test_evaluated"
    ]
    is False
)

print("Dataset/split checks: PASSED")
print("Temporal-result checks: PASSED")
print("Targeted-ablation checks: PASSED")
print("Permutation-importance checks: PASSED")
print("Operational-complexity checks: PASSED")
print("Feature-provenance checks: PASSED")
print("Bandwidth-augmentation checks: PASSED")
print("Image-feasibility checks: PASSED")
print()
print("EVIDENCE CONTENT VALIDATION PASSED")

In [ ]:
def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


records = []

for evidence_name, path in EVIDENCE.items():
    records.append({
        "evidence_name": evidence_name,
        "path": str(path),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

inventory = (
    pd.DataFrame(records)
    .sort_values("evidence_name")
    .reset_index(drop=True)
)

inventory_path = (
    FREEZE_DIR
    / "authoritative_evidence_inventory.csv"
)

inventory.to_csv(
    inventory_path,
    index=False,
)

freeze_record = {
    "version": "1.0",
    "created_utc": (
        datetime.now(timezone.utc)
        .isoformat()
    ),
    "status": (
        "revision_evidence_frozen"
    ),
    "evidence_file_count": len(records),
    "numerical_experiments_complete": True,
    "further_training_planned": False,
    "evidence_files": records,
    "scientific_decisions": {
        "temporal_generalization": (
            "Report grouped-random and "
            "chronological evaluation separately."
        ),
        "timestamp_features": (
            "Report timestamp-free experiments "
            "and targeted feature ablations."
        ),
        "interpretability": (
            "Treat heterogeneous native-attribution "
            "agreement as an empirical diagnostic, "
            "not method-independent equivalence. "
            "Use common permutation importance as "
            "a sensitivity analysis."
        ),
        "complexity": (
            "Use serialized reconstruction-bundle "
            "size and single-event CPU latency. "
            "Do not equate tree leaves with neural "
            "network parameters."
        ),
        "richer_features": (
            "Report bandwidth augmentation as a "
            "bounded architecture-dependent result."
        ),
        "image_comparison": (
            "Do not report unmatched archived CNN "
            "or ViT scores. A controlled comparison "
            "requires retraining with the frozen "
            "event-level split assignments."
        ),
    },
}

freeze_json_path = (
    FREEZE_DIR
    / "revision_evidence_freeze.json"
)

freeze_json_path.write_text(
    json.dumps(
        freeze_record,
        indent=2,
    )
    + "\n"
)

print(
    inventory[
        [
            "evidence_name",
            "bytes",
            "sha256",
        ]
    ].to_string(index=False)
)

print()
print("SAVED:", inventory_path)
print("SAVED:", freeze_json_path)
print()
print("REVISION EVIDENCE FREEZE WRITTEN")

In [ ]:
inventory_path = (
    FREEZE_DIR
    / "authoritative_evidence_inventory.csv"
)

freeze_json_path = (
    FREEZE_DIR
    / "revision_evidence_freeze.json"
)

assert inventory_path.is_file()
assert freeze_json_path.is_file()

frozen_inventory = pd.read_csv(
    inventory_path
)

with freeze_json_path.open() as handle:
    frozen_record = json.load(handle)

assert len(frozen_inventory) == 19

assert (
    frozen_record["evidence_file_count"]
    == 19
)

assert (
    frozen_record["status"]
    == "revision_evidence_frozen"
)

assert (
    frozen_record[
        "numerical_experiments_complete"
    ]
    is True
)

assert (
    frozen_record[
        "further_training_planned"
    ]
    is False
)

for row in frozen_inventory.itertuples(
    index=False
):
    path = Path(row.path)

    assert path.is_file(), (
        f"Frozen evidence disappeared: {path}"
    )

    observed_hash = sha256_file(path)

    assert observed_hash == row.sha256, (
        f"Hash mismatch: {path}"
    )

print(
    "READY:",
    inventory_path.name,
    f"({inventory_path.stat().st_size:,} bytes)",
)

print(
    "READY:",
    freeze_json_path.name,
    f"({freeze_json_path.stat().st_size:,} bytes)",
)

print()
print(
    "Frozen evidence files:",
    len(frozen_inventory),
)

print(
    "Status:",
    frozen_record["status"],
)

print(
    "Further training planned:",
    frozen_record[
        "further_training_planned"
    ],
)

print()
print("EVIDENCE-FREEZE COMPLETION CHECK PASSED")